# Module 01: GKE Standard Cluster & Accelerator Node Pools Setup
In this module, you will provision a GKE Standard Cluster with Shielded VM security flags and custom VPC networking.

### Learning Objectives:
1. Create a GKE Standard Cluster with Workload Identity, Shielded VM, and IP-Aliasing enabled.
2. Verify default CPU node pool (used for zero-quota multi-node CPU testing).
3. Optional: Add dedicated GPU Node Pool (NVIDIA L4/A100) or TPU Node Pool (v5e 2x4 slice).
4. Install Kubernetes JobSet Operator (`jobset.x-k8s.io`).



In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))
import config
cfg = config.load_config("../config.env")

PROJECT_ID = cfg["PROJECT_ID"]
REGION = cfg["REGION"]
ZONE = cfg["ZONE"]
CLUSTER_NAME = cfg["CLUSTER_NAME"]
NETWORK_NAME = cfg["NETWORK_NAME"]
SUBNET_NAME = cfg["SUBNET_NAME"]

GPU_NODE_POOL = cfg["GPU_NODE_POOL_NAME"]
GPU_MACHINE = cfg["GPU_MACHINE_TYPE"]
GPU_TYPE = cfg["GPU_TYPE"]
GPU_COUNT = cfg["GPU_COUNT_PER_NODE"]
GPU_NODES = cfg["GPU_NODE_COUNT"]

TPU_NODE_POOL = cfg["TPU_NODE_POOL_NAME"]
TPU_MACHINE = cfg["TPU_MACHINE_TYPE"]
TPU_TOPOLOGY = cfg["TPU_TOPOLOGY"]
TPU_NODES = cfg["TPU_NODE_COUNT"]

JOBSET_VERSION = cfg["JOBSET_VERSION"]



## 1. Create Base GKE Cluster with Shielded Nodes & IP-Alias


In [ ]:
!gcloud container clusters create {CLUSTER_NAME}     --zone={ZONE}     --release-channel=regular     --workload-pool={PROJECT_ID}.svc.id.goog     --enable-ip-alias     --network={NETWORK_NAME}     --subnetwork={SUBNET_NAME}     --cluster-secondary-range-name=pods-range     --services-secondary-range-name=services-range     --enable-shielded-nodes     --shielded-secure-boot     --shielded-integrity-monitoring     --num-nodes=2     --machine-type=e2-standard-4



In [ ]:
!gcloud container clusters get-credentials {CLUSTER_NAME} --zone={ZONE}


## 2. Install JobSet Operator


In [ ]:
print("Installing Kubernetes JobSet Controller...")
!kubectl apply --server-side -f https://github.com/kubernetes-sigs/jobset/releases/download/{JOBSET_VERSION}/manifests.yaml



## 3. Optional: Provision GPU Node Pool (If GPU Quota Available)


In [ ]:
# Uncomment below to provision GPU node pool:
# !gcloud container node-pools create {GPU_NODE_POOL} #     --cluster={CLUSTER_NAME} #     --zone={ZONE} #     --machine-type={GPU_MACHINE} #     --accelerator=type={GPU_TYPE},count={GPU_COUNT} #     --num-nodes={GPU_NODES} #     --shielded-secure-boot #     --shielded-integrity-monitoring

# !kubectl apply -f https://raw.githubusercontent.com/GoogleCloudPlatform/container-engine-accelerators/master/nvidia-driver-installer/ubuntu/daemonset-unified.yaml



## 4. Optional: Provision TPU Node Pool (If TPU Quota Available)


In [ ]:
# Uncomment below to provision TPU v5e slice pool:
# !gcloud container node-pools create {TPU_NODE_POOL} #     --cluster={CLUSTER_NAME} #     --zone={ZONE} #     --node-locations={ZONE} #     --machine-type={TPU_MACHINE} #     --tpu-topology={TPU_TOPOLOGY} #     --shielded-secure-boot #     --shielded-integrity-monitoring #     --num-nodes={TPU_NODES}



## 5. Verify Cluster Readiness


In [ ]:
!kubectl get nodes -o wide
!kubectl get crds | grep jobsets

